# Tier 2-D — who causes the demand-side stability price

Tier 1-D says *how much* of the nodal price $\pi_k$ is stability. It does not say **whose
corridor** the load at bus $k$ is tightening. This notebook does.

The implicit function theorem on the dynamic equality block $F(x,P_d)=0$ gives

$$\frac{dx}{dP_d} = -F_x^{-1}F_{P_d},\qquad
S_{P_d} = h_{P_d} + h_x\frac{dx}{dP_d},\qquad
\tilde\gamma^\top S_{P_d} + \sum_b \mu_b^\top S_{b,P_d} \;=\; \alpha^\top F_{P_d} \;=\; -\,\mathrm{stability}_k$$

**$h_{P_d} = 0$ and $b_{P_d} = 0$ here**, and for a structural reason rather than a
numerical one: the corridors bound $\delta$ and $\Delta\omega$, the state boxes bound
$E$, $\delta$ and $P_m$, and none of those rows contains load. Asserted below, not assumed.

## Why this is cheap

Every row carrying $P_d$ is **already inside the $F$ block** — `eq_const_Pbalance_tf/_tpf`
on FULL_BUS, `eq_const_Pe_tf/_tpf` on Kron — so no new `DynStateSpec` and no new calculus.
$F_x$ and its LU are the same ones the generator-side Tier 2 builds; $S_{P_d}$ costs one
extra back-substitution per bus.

## The one thing that is not free

$P_g$ is a JuMP variable, so $F_P$ is a slice of the exported Jacobian. $P_d$ is a
`Float64` inlined into the builders, so MOI has no column for it and $F_{P_d}$ must be
*computed* — that is `pd_coeffs.py`, shared with the Tier 1-D notebook so the two cannot
drift. Its coefficients were validated against a model-rebuild finite difference at
$1.4\times10^{-14}$ (FULL_BUS) and $5\times10^{-7}$ (Kron).

In [ ]:
import os, sys, pathlib, tomllib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.sparse.linalg import splu

# Same run resolution as the other KKT notebooks: KKT_RUN_DIR wins, else this repo's
# RESULTS; a root resolves to the newest run carrying a Tier-2 export.
RUN = pathlib.Path(os.environ.get("KKT_RUN_DIR", r"..\RESULTS"))
if RUN.is_dir() and not (RUN / "run_manifest.toml").exists():
    cand = sorted(p.parents[1] for p in RUN.glob("Results - */Sensitivity/sensitivity_meta.toml"))
    if not cand:
        raise SystemExit(f"No run under {RUN.resolve()} has a Sensitivity/ export. "
                         "Produce one with a driver that sets the Tier-2 post_solve_hook.")
    RUN = cand[-1]

SENS = RUN / "Sensitivity"
rd = lambda p: pd.read_csv(p, sep=";")

_here = pathlib.Path.cwd()
for _c in (_here, _here / "private" / "kkt", pathlib.Path(r"..\..\private\kkt")):
    if (_c / "pd_coeffs.py").is_file():
        sys.path.insert(0, str(_c.resolve()))
        break
else:
    raise SystemExit("pd_coeffs.py not found; run this notebook from kkt/")
import pd_coeffs

with open(RUN / "run_manifest.toml", "rb") as fh:
    M = tomllib.load(fh)
with open(SENS / "sensitivity_meta.toml", "rb") as fh:
    SM = tomllib.load(fh)

res = M["resolved"]
dynm = M.get("transient", {}).get("dyn_model", {})
NETWORK = dynm.get("network_form", "KRON_REDUCED")
ZIP_P = tuple(res.get("zip_load_p", dynm.get("zip_load_p", [1.0, 0.0, 0.0])))
N_TF, N_TPF = int(res["n_steps_fault"]), int(res["n_steps_postfault"])

# A per-load table refines the load model bus by bus; the balances were built from the
# demand-weighted mean of the loads at each bus, so that is the split F_Pd must use.
LOADS = (pd_coeffs.read_load_table(RUN, load_factor=float(M["run"]["load_factor"]))
         if NETWORK == "FULL_BUS" else None)
ZIP_BUS = pd_coeffs.zip_by_bus(LOADS, ZIP_P)

print(f"run        {RUN.name}")
print(f"case       {M['run']['case']} x lf {M['run']['load_factor']}   {SM['path']}")
print(f"zip_load_p {ZIP_P}  (Z, I, P)      steps {N_TF} fault + {N_TPF} post-fault")
print(f"block      F {SM['n_F_rows']}  x {SM['n_x_cols']}  square={SM['square']}")
print(f"corridors  delta {SM['n_h_rows']} rows (max|gamma| {SM['gamma_binds']:.4g})   "
      f"omega {SM.get('n_h_omega_rows', 0)} rows "
      f"(max|gamma| {SM.get('gamma_omega_binds', 0.0):.4g})")
print(f"status     {M['status'].get('termination_status')}")

assert SM["square"], "dynamic block is not square - S is undefined"
if SM["n_h_rows"] == 0 and SM.get("n_h_omega_rows", 0) == 0:
    raise SystemExit("Neither corridor was built - there is nothing to price.")

In [ ]:
# --- F_x, its LU, and the load column -------------------------------------------------
rows, cols = rd(SENS / "jacobian_rows.csv"), rd(SENS / "jacobian_cols.csv")
coo = rd(SENS / "jacobian_coo.csv")
J = sp.csr_matrix((coo.value, (coo.row - 1, coo.col - 1)),
                  shape=(len(rows), len(cols)))          # Julia is 1-indexed

idx = lambda df, blk, c: df.loc[df.block == blk, c].to_numpy() - 1
F_rows, x_cols = idx(rows, "F", "row"), idx(cols, "x", "col")

F_x = sp.csc_matrix(J[F_rows][:, x_cols])
lu = splu(F_x)                                            # the expensive step, once

F_Pd, BUS_IDS, placed = pd_coeffs.f_pd_matrix(
    RUN, rows, network=NETWORK, zip_p=ZIP_BUS, n_tf=N_TF, n_tpf=N_TPF)
dxdPd = -lu.solve(F_Pd)

alpha = rows.loc[rows.block == "F", "dual"].to_numpy()
rhs = alpha @ F_Pd                                        # = -stability_k

# h_Pd = 0 and b_Pd = 0 are STRUCTURAL: no corridor or state-box family is one of the
# families that carry load. Check it rather than trusting the claim.
pd_fams = set(pd_coeffs.PD_FAMILIES[NETWORK])
for blk_name in ("h", "h_lower", "h_omega", "h_omega_lower", "x_block"):
    clash = pd_fams & set(rows.loc[rows.block == blk_name, "family"])
    assert not clash, f"{blk_name} contains a load-carrying family {clash}: h_Pd is not 0"

print(f"F_x {F_x.shape} nnz {F_x.nnz}    F_Pd {F_Pd.shape}    coefficients placed {placed}")
print(f"buses {BUS_IDS}")

In [ ]:
# --- the channels ---------------------------------------------------------------------
def corridor_rows(block, families):
    """Order rows the way the exporter did: declared family first, then container key.
    Jacobian row order instead gives a row-permuted S. A scalar identity cannot detect
    that - a sum is invariant to it - but every individual sensitivity would then be
    attributed to the wrong (machine, step)."""
    rank = {f: i for i, f in enumerate(families)}
    sub = rows[rows.block == block].copy()
    sub["rank"] = sub.family.map(rank)
    return sub.sort_values(["rank", "key1", "key2"], kind="stable")

CH, charges = {}, {}
for nm, bu, bl, fu, fl in (
        ("delta", "h", "h_lower", SM["families_h"], SM["families_h_lower"]),
        ("omega", "h_omega", "h_omega_lower",
         SM.get("families_h_omega", []), SM.get("families_h_omega_lower", []))):
    up = corridor_rows(bu, fu)
    if up.empty:
        continue
    lo = corridor_rows(bl, fl)
    assert (up[["key1", "key2"]].to_numpy() == lo[["key1", "key2"]].to_numpy()).all(), \
        f"{nm}: upper/lower rows do not pair up on (machine, step)"
    r_up = up.row.to_numpy() - 1
    S = J[r_up][:, x_cols] @ dxdPd                        # h_Pd = 0, asserted above
    gam = up.dual.to_numpy() - lo.dual.to_numpy()         # gamma tilde = G+ - G-
    CH[nm] = dict(S=S, gam=gam, key=up[["key1", "key2"]].to_numpy())
    charges[nm] = gam @ S

# Every other active inequality on a state, charged as (mu' b_x) @ dxdPd. Never
# materialise S_b: the FULL_BUS voltage floor alone is one row per bus per step.
blk = rows[rows.block == "x_block"]
charge_blocks = ((blk.dual.to_numpy() @ J[blk.row.to_numpy() - 1][:, x_cols]) @ dxdPd
                 if len(blk) else np.zeros(len(BUS_IDS)))

charge_total = sum(charges.values()) + charge_blocks

# Stationarity in x from EVERY row against its own dual, with no reference to the
# F / h / x_block partition. The identity residual collapses to exactly r' @ dxdPd, so
# the gap between the two is the charge of rows the partition missed and nothing else.
r_true = J[:, x_cols].T @ rows.dual.to_numpy()
explained = r_true @ dxdPd
residual = charge_total - rhs
unexplained = residual - explained

# Cross-check against Tier 1-D. Both use the same coefficients, so this does not test
# their scale - but Tier 1-D contracts in PHYSICS space and this notebook in JACOBIAN-ROW
# space, so any row-join or fault/post-fault rebase error breaks the match. This is the
# check that catches the step that can actually go wrong.
t1_path = RUN / "kkt_Pd_prices.csv"
if t1_path.exists():
    stab = rd(t1_path).set_index("bus").stability.reindex(BUS_IDS).to_numpy()
    print(f"indexing check   max |alpha'F_Pd + stability| = {np.abs(rhs + stab).max():.3e}"
          "    <- run KKT_Pd_Duals.ipynb first if this file is missing")
else:
    stab = np.full(len(BUS_IDS), np.nan)
    print("indexing check   SKIPPED (no kkt_Pd_prices.csv; run KKT_Pd_Duals.ipynb first)")

out = pd.DataFrame({"bus": BUS_IDS, "PD_MW": np.nan, **{f"charge_{k}": v for k, v in charges.items()},
                    "charge_blocks": charge_blocks, "charge_total": charge_total,
                    "-stability": rhs, "residual": residual,
                    "explained": explained, "unexplained": unexplained}).set_index("bus")
buses_rep = rd(RUN / "Dispatch" / "CSV" / "buses_report.csv").set_index("BUS")
out["PD_MW"] = buses_rep["PD_MW"].reindex(out.index)
display(out.round(6))

print(f"max |residual|    = {np.abs(residual).max():.3e}"
      "   <- solver dual tolerance x ||dx/dP||, grows with n_x; NOT an error")
print(f"    explained     = {np.abs(explained).max():.3e}")
print(f"    UNEXPLAINED   = {np.abs(unexplained).max():.3e}"
      "   <- the identity; this is the one that must be ~0")

charge_scale = max((np.abs(v).max() for v in charges.values()), default=0.0)
if charge_scale < 1e-4:
    print("\n*** NOT A DISCRIMINATING TEST ***")
    print(f"    max |charge| = {charge_scale:.2e} EUR/MW - no corridor carries a price in")
    print("    this run, so the identity is verified at zero magnitude. Still a real")
    print("    check of S, but it proves nothing about attribution.")
else:
    print(f"\n    DISCRIMINATING: max |charge| = {charge_scale:.3e} EUR/MW")

out.to_csv(SENS / "charges_Pd_by_channel.csv", sep=";")
print(f"written: {SENS / 'charges_Pd_by_channel.csv'}")

In [ ]:
# --- the attribution: which machine's corridor does each bus's load tighten? -----------
# This is what Tier 2-D adds over Tier 1-D. Tier 1-D gives one stability number per bus;
# here that number is split across the machines whose corridor the load actually moves.
#
# Figures are displayed only: nbconvert embeds them in the HTML report, so writing
# standalone .svg files beside it just duplicates the same picture into the run folder.
for nm, c in CH.items():
    S, gam, key = c["S"], c["gam"], c["key"]
    gens = np.unique(key[:, 0])
    # charge contributed to bus k by machine g's corridor rows, summed over time
    A = np.vstack([gam[key[:, 0] == g] @ S[key[:, 0] == g] for g in gens])
    tab = pd.DataFrame(A, index=[f"corridor G{g}" for g in gens],
                       columns=[f"bus {b}" for b in BUS_IDS])
    print(f"({nm}) causer attribution  [EUR/MW]   row = whose corridor, column = whose load")
    display(tab.round(4))
    print(f"      column sums reproduce charge_{nm}: "
          f"max diff {np.abs(A.sum(axis=0) - charges[nm]).max():.3e}\n")

    fig, ax = plt.subplots(figsize=(1.05 * len(BUS_IDS) + 3, 0.55 * len(gens) + 2.4))
    lim = np.abs(A).max() or 1.0
    im = ax.imshow(A, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(len(BUS_IDS)), [str(b) for b in BUS_IDS])
    ax.set_yticks(range(len(gens)), [f"G{g}" for g in gens])
    ax.set_xlabel("load bus"); ax.set_ylabel("corridor of machine")
    ax.set_title(f"Tier 2-D {nm}: stability charge by (causing load bus, constrained machine)"
                 f"\n{SM['path']} - {RUN.name}")
    for (r_, c_), v in np.ndenumerate(A):
        ax.text(c_, r_, f"{v:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.85, label="EUR/MW")
    fig.tight_layout()
    plt.show()

# The identity, drawn. The two bars must COINCIDE, not mirror.
fig, ax = plt.subplots(figsize=(1.0 * len(BUS_IDS) + 3, 4.0))
x = np.arange(len(BUS_IDS))
ax.bar(x - 0.2, charge_total, width=0.4, label="total causer charge")
ax.bar(x + 0.2, rhs, width=0.4, label=r"$-$stability (Tier 1-D)")
ax.axhline(0, color="gray", lw=0.6)
ax.set_xticks(x, [str(b) for b in BUS_IDS])
ax.set_xlabel("bus"); ax.set_ylabel("EUR/MW")
ax.set_title("the identity: total charge = -stability")
ax.legend(); fig.tight_layout()
plt.show()

## Reading it

**The identity is not an approximation.** It follows from stationarity in the state,
$F_x^\top\alpha + h_x^\top\tilde\gamma + \sum_b b_x^\top\mu_b = 0$. Substituting it into
$\tilde\gamma^\top S_{P_d} = \tilde\gamma^\top h_x(-F_x^{-1}F_{P_d})$ collapses the
factorisation and leaves $\alpha^\top F_{P_d}$. The residual measures solver tolerance and
float64 conditioning, nothing else — read **`unexplained`**, never the raw residual, which
is Ipopt's dual tolerance amplified by $\lVert dx/dP\rVert$ and grows with $n_x$.

**What the attribution means.** Row $g$, column $k$ is the part of bus $k$'s stability price
that exists because load there moves machine $g$'s corridor. It is the demand-side
counterpart of the generator causer-pays table: the same money, now split by *who is
constrained* rather than *who is charged*.

**Both signs are physical.** A positive charge means load at that bus tightens the
corridor and pays for it. A negative one means it helps — under a constant-impedance ZIP
model a load near the fault acts as a braking resistance in the post-fault window. Buses
at or near the faulted bus routinely come out negative, and the total then *reduces* their
nodal price.

**The corridor must bind.** If $\tilde\gamma\approx 0$ the identity holds at zero and the
attribution is noise. The banner says whether the run is discriminating. On 9-bus Kron the
swing is reshaped through $Q$ and $V$, which cost nothing, so both corridors go from slack
to infeasible without ever being priced — that is a property of the case, not a bug.

**Kron overstates the stability channel** relative to FULL_BUS (33–53 % on 9-bus). Compare
the split, never the bottom line alone: at the faulted bus the totals can agree while both
channels are individually wrong, because $\lambda$ and the stability term partly cancel.